# Vietnamese Credit Card Fraud Detection - EDA
## Phase 1: Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

In [ ]:
# Load dataset
train_path = '../dataset/fraudTrain.csv'
test_path = '../dataset/fraudTest.csv'

print('Loading training data...')
train_df = pd.read_csv(train_path)
print(f'Training data loaded: {train_df.shape}')

print('\nLoading test data...')
test_df = pd.read_csv(test_path)
print(f'Test data loaded: {test_df.shape}')

In [ ]:
# 1. Check shape
print('='*50)
print('1. DATASET SHAPE')
print('='*50)
print(f'Training set: {train_df.shape}')
print(f'Test set: {test_df.shape}')
print(f'\nColumns ({len(train_df.columns)}):')
for col in train_df.columns:
    print(f'  - {col}')

In [ ]:
# 2. Check missing values
print('='*50)
print('2. MISSING VALUES')
print('='*50)
print('\nTraining set:')
missing_train = train_df.isnull().sum()
missing_train = missing_train[missing_train > 0]
if len(missing_train) > 0:
    print(missing_train)
else:
    print('No missing values!')

print('\nTest set:')
missing_test = test_df.isnull().sum()
missing_test = missing_test[missing_test > 0]
if len(missing_test) > 0:
    print(missing_test)
else:
    print('No missing values!')

In [ ]:
# 3. Check duplicates
print('='*50)
print('3. DUPLICATES')
print('='*50)
print(f'Training duplicates: {train_df.duplicated().sum()}')
print(f'Test duplicates: {test_df.duplicated().sum()}')

In [ ]:
# 4. Check datatypes
print('='*50)
print('4. DATATYPES')
print('='*50)
print('\nTraining set:')
print(train_df.dtypes)

In [ ]:
# 5. Target distribution
print('='*50)
print('5. TARGET DISTRIBUTION (is_fraud)')
print('='*50)
print('\nTraining set:')
print(train_df['is_fraud'].value_counts())
print(f'\nFraud rate: {train_df["is_fraud"].mean()*100:.2f}%')

print('\nTest set:')
print(test_df['is_fraud'].value_counts())
print(f'\nFraud rate: {test_df["is_fraud"].mean()*100:.2f}%')

In [ ]:
# Visualization 1: Fraud vs Normal
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training set
train_counts = train_df['is_fraud'].value_counts()
axes[0].pie(train_counts.values, labels=['Normal', 'Fraud'], autopct='%1.1f%%', 
           colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Training Set: Fraud vs Normal')

# Test set
test_counts = test_df['is_fraud'].value_counts()
axes[1].pie(test_counts.values, labels=['Normal', 'Fraud'], autopct='%1.1f%%',
           colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Test Set: Fraud vs Normal')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Transaction amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
train_df[train_df['is_fraud']==0]['amt'].hist(bins=50, alpha=0.6, label='Normal', ax=axes[0])
train_df[train_df['is_fraud']==1]['amt'].hist(bins=50, alpha=0.6, label='Fraud', ax=axes[0])
axes[0].set_title('Training: Amount Distribution')
axes[0].set_xlabel('Amount (USD)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].set_xlim(0, 2000)

# Test set
test_df[test_df['is_fraud']==0]['amt'].hist(bins=50, alpha=0.6, label='Normal', ax=axes[1])
test_df[test_df['is_fraud']==1]['amt'].hist(bins=50, alpha=0.6, label='Fraud', ax=axes[1])
axes[1].set_title('Test: Amount Distribution')
axes[1].set_xlabel('Amount (USD)')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].set_xlim(0, 2000)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Fraud by Category
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

train_cat = train_df.groupby('category')['is_fraud'].mean().sort_values(ascending=False)
train_cat.plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Training: Fraud Rate by Category')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Fraud Rate')
axes[0].tick_params(axis='x', rotation=45)

test_cat = test_df.groupby('category')['is_fraud'].mean().sort_values(ascending=False)
test_cat.plot(kind='bar', ax=axes[1], color='skyblue')
axes[1].set_title('Test: Fraud Rate by Category')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Fraud Rate')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 4: Fraud by State
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

train_state = train_df.groupby('state')['is_fraud'].mean().sort_values(ascending=False).head(15)
train_state.plot(kind='bar', ax=axes[0], color='lightcoral')
axes[0].set_title('Training: Fraud Rate by State (Top 15)')
axes[0].set_xlabel('State')
axes[0].set_ylabel('Fraud Rate')
axes[0].tick_params(axis='x', rotation=45)

test_state = test_df.groupby('state')['is_fraud'].mean().sort_values(ascending=False).head(15)
test_state.plot(kind='bar', ax=axes[1], color='lightskyblue')
axes[1].set_title('Test: Fraud Rate by State (Top 15)')
axes[1].set_xlabel('State')
axes[1].set_ylabel('Fraud Rate')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 5: Fraud by Hour of Day
train_df['trans_date_trans_time'] = pd.to_datetime(train_df['trans_date_trans_time'])
train_df['hour'] = train_df['trans_date_trans_time'].dt.hour

hourly_fraud = train_df.groupby('hour')['is_fraud'].mean()

plt.figure(figsize=(12, 5))
plt.plot(hourly_fraud.index, hourly_fraud.values, marker='o', linewidth=2, color='darkred')
plt.fill_between(hourly_fraud.index, hourly_fraud.values, alpha=0.3, color='red')
plt.title('Fraud Rate by Hour of Day')
plt.xlabel('Hour (0-23)')
plt.ylabel('Fraud Rate')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualization 6: Correlation Analysis
numeric_cols = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'is_fraud']
corr_matrix = train_df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Summary Statistics
print('='*50)
print('SUMMARY STATISTICS')
print('='*50)

print('\nTraining set:')
print(f'  Shape: {train_df.shape}')
print(f'  Memory: {train_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print(f'  Fraud cases: {train_df["is_fraud"].sum():,}')
print(f'  Normal cases: {(train_df["is_fraud"]==0).sum():,}')
print(f'  Fraud rate: {train_df["is_fraud"].mean()*100:.2f}%')
print(f'  Categories: {train_df["category"].nunique()}')
print(f'  States: {train_df["state"].nunique()}')
print(f'  Merchants: {train_df["merchant"].nunique()}')

print('\nTest set:')
print(f'  Shape: {test_df.shape}')
print(f'  Fraud cases: {test_df["is_fraud"].sum():,}')
print(f'  Normal cases: {(test_df["is_fraud"]==0).sum():,}')
print(f'  Fraud rate: {test_df["is_fraud"].mean()*100:.2f}%')

print('\n' + '='*50)
print('EDA COMPLETE!')
print('='*50)